In [1]:
import os
import fitz  # PyMuPDF
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("1. Setting up Vector Database...")
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# We use persist_directory to save the database permanently to your hard drive
persist_dir = "../data/chroma_db_massive"
vector_db = Chroma(
    collection_name="massive_physics_db",
    embedding_function=embedding_model,
    persist_directory=persist_dir
)

# A robust chunker for massive text documents
def chunk_text(text, max_chars=1000):
    # Split the text by double newlines to respect paragraphs
    paragraphs = text.split('\n\n')
    chunks = []
    current_chunk = ""
    
    for p in paragraphs:
        if len(current_chunk) + len(p) < max_chars:
            current_chunk += p + "\n"
        else:
            if current_chunk.strip():
                chunks.append(current_chunk.strip())
            current_chunk = p + "\n"
            
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    return chunks

print("2. Reading and Processing PDFs...")
pdf_folder = "../data/raw/"
all_chunks = []

# Check if the folder exists and process all PDFs inside
if not os.path.exists(pdf_folder):
    print(f"Error: Folder {pdf_folder} not found!")
else:
    for filename in os.listdir(pdf_folder):
        if filename.endswith(".pdf"):
            print(f"-> Processing {filename}...")
            file_path = os.path.join(pdf_folder, filename)
            
            # Open PDF with PyMuPDF
            doc = fitz.open(file_path)
            full_text = ""
            
            # Read every single page
            for page_num in range(len(doc)):
                page = doc.load_page(page_num)
                full_text += page.get_text("text") + "\n\n"
                
            print(f"   Extracted {len(doc)} pages. Chunking text...")
            file_chunks = chunk_text(full_text)
            all_chunks.extend(file_chunks)
            print(f"   Created {len(file_chunks)} chunks from {filename}.")

print(f"\n3. Total chunks to insert: {len(all_chunks)}")
print("4. Inserting chunks into ChromaDB...")
print("   (This might take a few minutes as your laptop calculates the vectors!)")

# Insert in batches to prevent laptop memory overload
batch_size = 500
for i in range(0, len(all_chunks), batch_size):
    batch = all_chunks[i:i+batch_size]
    vector_db.add_texts(texts=batch)
    print(f"   Inserted batch {i} to {min(i+batch_size, len(all_chunks))} / {len(all_chunks)}...")

print("\nDONE! All PDF Volumes are now successfully embedded in the persistent database.")

1. Setting up Vector Database...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

C:\Users\User\AppData\Local\Temp\ipykernel_18660\1111001377.py:11: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(


2. Reading and Processing PDFs...
-> Processing university-physics-volume-1.pdf...
   Extracted 958 pages. Chunking text...
   Created 936 chunks from university-physics-volume-1.pdf.
-> Processing university-physics-volume-2.pdf...
   Extracted 780 pages. Chunking text...
   Created 760 chunks from university-physics-volume-2.pdf.
-> Processing university-physics-volume-3.pdf...
   Extracted 597 pages. Chunking text...
   Created 583 chunks from university-physics-volume-3.pdf.

3. Total chunks to insert: 2279
4. Inserting chunks into ChromaDB...
   (This might take a few minutes as your laptop calculates the vectors!)
   Inserted batch 0 to 500 / 2279...
   Inserted batch 500 to 1000 / 2279...
   Inserted batch 1000 to 1500 / 2279...
   Inserted batch 1500 to 2000 / 2279...
   Inserted batch 2000 to 2279 / 2279...

DONE! All PDF Volumes are now successfully embedded in the persistent database.
